# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [3]:
df = pd.read_csv("data/AviationData.csv", low_memory=False, encoding='latin-1')
print("Data shape:", df.shape)
df.head()
df.info()
df.describe(include='all')

Data shape: (88889, 31)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make          

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
count,88889,88889,88889,88889,88837,88663,34382,34373,50132,52704,...,82697,16648,77488.000000,76379.000000,76956.000000,82977.000000,84397,61724,82505,75118
unique,87951,2,88863,14782,27758,219,25589,27154,10374,24870,...,26,13590,NaN,NaN,NaN,NaN,4,12,17074,2924
top,20001212X19172,Accident,CEN22LA149,1984-06-30,"ANCHORAGE, AK",United States,332739N,0112457W,NONE,Private,...,Personal,Pilot,NaN,NaN,NaN,NaN,VMC,Landing,Probable Cause,25-09-2020
freq,3,85015,2,25,434,82248,19,24,1488,240,...,49448,258,NaN,NaN,NaN,NaN,77303,15428,61754,17019
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.647855,0.279881,0.357061,5.325440,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,5.485960,1.544084,2.235625,27.913634,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.000000,0.000000,0.000000,1.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.000000,0.000000,0.000000,2.000000,NaN,NaN,NaN,NaN


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [4]:
df = df[df['Aircraft.Category'].astype(str).str.lower() == 'airplane']
df = df[df['Amateur.Built'] == 'No']

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [5]:
df['Total.Fatal.Injuries'] = df['Total.Fatal.Injuries'].fillna(0)
df['Total.Serious.Injuries'] = df['Total.Serious.Injuries'].fillna(0)
df['Total.Minor.Injuries'] = df['Total.Minor.Injuries'].fillna(0)
df['Total.Uninjured'] = df['Total.Uninjured'].fillna(0)

df['Total.Passengers'] = (
    df['Total.Fatal.Injuries'] +
    df['Total.Serious.Injuries'] +
    df['Total.Minor.Injuries'] +
    df['Total.Uninjured']
)

df['Serious_Or_Fatal_Fraction'] = np.where(
    df['Total.Passengers'] > 0,
    (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total.Passengers'],
    np.nan
)

df['Aircraft.damage'] = df['Aircraft.damage'].fillna('Unknown')
df['Is_Destroyed'] = np.where(df['Aircraft.damage'] == 'Destroyed', 1, 0)

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [6]:
print("Unique values in Aircraft.damage:")
print(df['Aircraft.damage'].unique())

print("\nValue counts:")
print(df['Aircraft.damage'].value_counts(dropna=False))

print("\nNumber of nulls:", df['Aircraft.damage'].isnull().sum())

Unique values in Aircraft.damage:
['Substantial' 'Destroyed' 'Minor' 'Unknown']

Value counts:
Aircraft.damage
Substantial    18981
Destroyed       3144
Unknown         1367
Minor            925
Name: count, dtype: int64

Number of nulls: 0


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [7]:
df['Make'] = df['Make'].astype(str).str.upper().str.strip()
make_counts = df['Make'].value_counts()
print("Unique makes:", len(make_counts))
print(make_counts.head(20))

valid_makes = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(valid_makes)]
print("Shape after filtering makes:", df.shape)

Unique makes: 1128
Make
CESSNA                8443
PIPER                 4700
BEECH                 1684
BOEING                1309
MOONEY                 417
BELLANCA               281
GRUMMAN                249
AIRBUS                 243
MAULE                  232
AERONCA                227
AIR TRACTOR            224
CIRRUS DESIGN CORP     220
AIR TRACTOR INC        219
CHAMPION               169
LUSCOMBE               163
EMBRAER                155
STINSON                145
CIRRUS                 137
NORTH AMERICAN         118
MCDONNELL DOUGLAS      113
Name: count, dtype: int64
Shape after filtering makes: (20778, 34)


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [8]:
df = df.dropna(subset=['Model'])
df['Model'] = df['Model'].astype(str).str.upper().str.strip()
df['Make_Model'] = df['Make'] + "-" + df['Model']
model_counts = df['Make_Model'].value_counts()
print("Unique models:", len(model_counts))
print(model_counts.head(10))

Unique models: 2405
Make_Model
CESSNA-172         866
CESSNA-152         448
BOEING-737         403
CESSNA-182         344
CESSNA-172N        314
CESSNA-172S        276
PIPER-PA28         273
CESSNA-150         255
CESSNA-180         237
PIPER-PA-28-140    229
Name: count, dtype: int64


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [9]:
df['Engine.Type'] = df['Engine.Type'].fillna('Unknown')
df['Weather.Condition'] = df['Weather.Condition'].fillna('Unknown')
df['Number.of.Engines'] = pd.to_numeric(df['Number.of.Engines'], errors='coerce').fillna(0)
df['Purpose.of.flight'] = df['Purpose.of.flight'].fillna('Unknown')
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].fillna('Unknown')

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [10]:
threshold = 20000
cols_to_keep = df.columns[df.count() >= threshold].tolist()
df = df[cols_to_keep]
print("Columns kept:", cols_to_keep)
print("New shape:", df.shape)

Columns kept: ['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location', 'Country', 'Injury.Severity', 'Aircraft.damage', 'Aircraft.Category', 'Registration.Number', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description', 'Purpose.of.flight', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Total.Passengers', 'Is_Destroyed', 'Make_Model']
New shape: (20764, 26)


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [11]:
df.to_csv("cleaned_aviation_data.csv", index=False)
print("Cleaned data saved to 'cleaned_aviation_data.csv'")

Cleaned data saved to 'cleaned_aviation_data.csv'
